In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import nltk
import re
from wordcloud import WordCloud
from rouge_score import rouge_scorer

from nltk.tokenize import sent_tokenize, word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords

# Please uncomment if you want to download
# nltk.download('punkt')
# nltk.download('punkt_tab')      
# nltk.download('wordnet')    
# nltk.download('omw-1.4') 
# nltk.download('averaged_perceptron_tagger_eng') 

In [ ]:
try:
    df = pd.read_csv('filtered_data.csv', nrows=10000)
    df.to_csv('10000only.csv', index=False)
    display(df)
except:
    print("an error has occured")


# Data Exploration

In [ ]:
df['article_len'] = df['article'].apply(lambda x: len(x.split()))
plt.figure(figsize=(10,6))
plt.hist(df['article_len'], bins=50)
plt.title("Article Length Distribution")
plt.xlabel("Number of Words")
plt.ylabel("Frequency")
plt.show()

In [ ]:
df['summary_len'] = df['highlights'].apply(lambda x: len(x.split()))
plt.figure(figsize=(10,6))
plt.hist(df['summary_len'], bins=50)
plt.title("Summary Length Distribution")
plt.xlabel("Number of Words")
plt.ylabel("Frequency")
plt.show()

In [ ]:
plt.figure(figsize=(10,6))
plt.scatter(df['article_len'], df['summary_len'],s=10)
plt.title("Article Length vs Summary Length")
plt.xlabel("Article Length")
plt.ylabel("Summary Length")
plt.show()

In [ ]:
# simple cleaning for visualization
def clean_text(text):
    words = text.lower().split()
    words = [w for w in words if w.isalpha()]
    return " ".join(words)

articles_text = " ".join(df['article'].apply(clean_text))
highlights_text = " ".join(df['highlights'].apply(clean_text))

wordcloud_articles = WordCloud(
    width=800,
    height=400,
    background_color='white'
).generate(articles_text)

wordcloud_highlights = WordCloud(
    width=800,
    height=400,
    background_color='white',
    colormap='viridis'
).generate(highlights_text)


fig, ax = plt.subplots(1, 2, figsize=(15,6))

ax[0].imshow(wordcloud_articles, interpolation='bilinear')
ax[0].set_title("Articles")
ax[0].axis("off")

ax[1].imshow(wordcloud_highlights, interpolation='bilinear')
ax[1].set_title("Highlights")
ax[1].axis("off")

plt.show()

In [ ]:
df_clean = pd.DataFrame()
df_clean = df.drop(columns=['id','article_len','summary_len']).copy()

In [ ]:
df_clean

In [ ]:
df["article"][145]

In [ ]:
# remove bracketed publisher info
pattern1 = re.compile(
    r'^(?:[a-z,]+\s+){0,3}\((?:[a-z\.]+\s*){1,2}\)(?:\s+--\s+)?',
    re.IGNORECASE
)

# remove bylines, social media follows , publication dates, and update timestamps
pattern2 = re.compile(
    # r'^(?:By\s+.*?[a-z,\.@ ]+?\s+\.\s+)?(?:follow.*?\s*\.\s*)?(?:PUBLISHED:.*?\|)?(?:.*?UPDATED:\s\..*?\d{1,2}:\d{2}\s+[a-z\.]{2,5},\s*\d{1,2}\s+\w+\s+\d{4}\s*\.\s*)?',
    # r'^(?:By\s+.*?(?:[\w,\.@]+?\s+\.\s+){0,3})?(?:PUBLISHED:.*?\|)?(?:.*?UPDATED:\s\..*?\d{1,2}:\d{2}\s+[a-z\.]{2,5},\s*\d{1,2}\s+\w+\s+\d{4}\s*\.\s*)?',
    r'^(?:By\s+.*?(?:(?:[\w,\.@\s\[\]\']+){0,3}\s+\.\s+){0,3})?(?:PUBLISHED:.*?\|)?(?:.*?UPDATED:\s\..*?\d{1,2}:\d{2}\s+[a-z\.]{2,5},\s*\d{1,2}\s+\w+\s+\d{4}\s+\.\s+)?',
    re.IGNORECASE
)

# remove "Last updated" lines
pattern3 = re.compile(
    r'^Last updated.*?\s*\.\s*',
    re.IGNORECASE
)

In [ ]:
def remove_prefix(text):
    text = pattern1.sub('', text)
    text = pattern2.sub('', text)
    text = pattern3.sub('', text)
    return text


df_clean["article"] = df_clean["article"].apply(remove_prefix)
df_clean

# regex debug

In [ ]:
pattern1.findall("(EW.com) -- ")

In [ ]:
pattern2.findall(df["article"][145])

In [ ]:
df_clean["article"][0]

In [ ]:
df_clean["article"][9]

In [ ]:
df_clean["article"][20]

In [ ]:
df_clean["article"][31]

In [ ]:
df_clean["article"][37]

In [ ]:
df_clean["article"][83]

In [ ]:
df_clean["article"][100]

In [ ]:
df_clean["article"][119]

In [ ]:
df_clean["article"][145]

In [ ]:
df_clean["article"][166]

In [ ]:
df_clean["article"][188]

In [ ]:
df_clean["article"][271]

In [ ]:
df_clean["article"][374]

In [ ]:
df_clean["article"][488]

In [ ]:
df_clean["article"][9997]

In [ ]:
df_clean.to_csv('cleaned_data.csv', index=False)

In [ ]:
# Load cleaned data directly to save time
df_clean = pd.read_csv('cleaned_data.csv')

# Linear Regression

In [ ]:
lemmatizer= WordNetLemmatizer()
stop_words = set(stopwords.words('english'))    

In [ ]:
def preprocess(text):
    tokens = word_tokenize(text.lower())
    tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
        if word.isalpha() and word not in stop_words
    ]
    return tokens

In [ ]:
scorer = rouge_scorer.RougeScorer(['rouge1'])
def prepare_data(df):
    X = []
    y = []

    for _, row in df.iterrows():

        article_sentences = sent_tokenize(row['article'].lower())
        highlights_tokens =  preprocess(row['highlights'])

        highlights_text = " ".join(highlights_tokens)

        for sentence in article_sentences:

            words = preprocess(sentence)
            sentence_text = " ".join(words)

            # ROUGE similarity score
            score = scorer.score(sentence_text, highlights_text)['rouge1'].fmeasure

            X.append(sentence_text)
            y.append(score)

    return X, y

In [ ]:
sentences, labels = prepare_data(df_clean)
vectorizer = TfidfVectorizer()
X_vectors = vectorizer.fit_transform(sentences)

In [ ]:
X_vectors.shape

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_vectors, labels, test_size=0.2, random_state=42)

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)

In [ ]:
def summarize_article(article, model, vectorizer, num_sentences=3):

    source_sentences = sent_tokenize(article)

    pre_sentences = [" ".join(preprocess(sentence)) for sentence in source_sentences]
    sentence_vectors = vectorizer.transform(pre_sentences)

    scores = model.predict(sentence_vectors)

    ranked_indices = np.argsort(scores)[::-1]

    # Select top-k sentences
    top_indices = sorted(ranked_indices[:num_sentences])

    summary = " ".join([source_sentences[i] for i in top_indices])

    return summary

In [ ]:
article = int(input("Enter the article index (0-9999): "))
summary_sents = int(input("Enter the number of sentences for the summary: "))
test_article = df_clean['article'].iloc[article]
actual_highlight = df_clean['highlights'].iloc[article]

generated_summary = summarize_article(test_article, lr, vectorizer, summary_sents)

print("\n------- Original Article Snippet ---")
print(test_article[:300] + "...")
print("\n--- Actual Highlight -------")
print(actual_highlight)
print("\n--- Linear Regression Predicted Summary ---")
print(generated_summary)